**Linear SVM**

Install libraire

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



In [ ]:
# chargement du dataset
df = pd.read_csv("../data/labelled_topics/dataset_avis.csv")
df.head()

In [4]:
# colonnes cibles(multilabel)
labels = ["qualité produit", "service livraison", "service client"]


In [5]:
# Données texte (features)
X = df["clean_comment"]



In [6]:
# Données cibles
y = df[labels].values


In [7]:
#TF-IDF vectorization

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(X)

In [8]:
#Séparation Train / Test
# Nous séparons le dataset en ensembles d'entraînement et de test (80/20).
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
# Modèle SVM Multilabel
# nous entraînons un classifieur LinearSVC dans un schéma OneVsRest pour gérer les labels multiples.
# LinearSVC as base model
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

# multilable avec OneVsRest
clf = OneVsRestClassifier(svm_model)

In [ ]:
#Validation croisée (F1 micro)
#Évaluation du modèle sur 5 folds avec le F1-score micro.
scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_tfidf,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro per fold :", cv_scores)
print("F1 micro average  :", cv_scores.mean())
print("Standard deviation :", cv_scores.std())

Entraînement et évaluation sur le test set

In [11]:
# Train on train set
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)



# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# # Scores globaux
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")
#  Évaluation sur le jeu de test
print(classification_report(y_test, y_pred, target_names=labels))


Label 'qualité produit': Accuracy = 0.733, F1-score = 0.693
Label 'service livraison': Accuracy = 0.652, F1-score = 0.648
Label 'service client': Accuracy = 0.830, F1-score = 0.486

F1 micro    : 0.6416
F1 weighted : 0.6448
                   precision    recall  f1-score   support

  qualité produit       0.66      0.73      0.69       226
service livraison       0.68      0.62      0.65       284
   service client       0.42      0.59      0.49        75

        micro avg       0.63      0.66      0.64       585
        macro avg       0.59      0.64      0.61       585
     weighted avg       0.64      0.66      0.64       585
      samples avg       0.62      0.67      0.63       585



c:\Users\vires\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Matrices de confusion

In [12]:

labels = ["qualité produit", "service livraison", "service client"]
for i, label in enumerate(labels):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    print(f"Confusion matrix for {label}:")
   # print(cm)
    cm_df = pd.DataFrame(cm,index = ["Vrai 0","Vrai 1"],columns = ["Prédit 0","Prédit 1"])
    print(cm_df)

Confusion matrix for qualité produit:
        Prédit 0  Prédit 1
Vrai 0       235        85
Vrai 1        61       165
Confusion matrix for service livraison:
        Prédit 0  Prédit 1
Vrai 0       181        81
Vrai 1       109       175
Confusion matrix for service client:
        Prédit 0  Prédit 1
Vrai 0       409        62
Vrai 1        31        44


Évaluation sur 100_avis_annote.csv

In [ ]:
#Évaluation sur 100_avis_annote.csv
df_avis = pd.read_csv("../data/test_dataset/100_avis_annote.csv", sep=";")
df_avis.head()

In [14]:
# TF-IDF
X_avis_tfidf = tfidf.transform(df_avis['clean_comment'])
y_avis_true = df_avis[["qualité produit", "service livraison", "service client"]].values

In [15]:
# Prédiction
y_avis_pred = clf.predict(X_avis_tfidf)

In [16]:
# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_avis_true[:, i] == y_avis_pred[:, i])
    f1 = f1_score(y_avis_true[:, i],y_avis_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_avis_true, y_avis_pred, average="micro")
f1_weighted = f1_score(y_avis_true, y_avis_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")

Label 'qualité produit': Accuracy = 0.530, F1-score = 0.254
Label 'service livraison': Accuracy = 0.540, F1-score = 0.361
Label 'service client': Accuracy = 0.590, F1-score = 0.128

F1 micro    : 0.2637
F1 weighted : 0.2409


In [17]:
# Classification report
print("SVM Results on the 100 Manual Labels:")
print(classification_report(y_avis_true, y_avis_pred, target_names=labels))

SVM Results on the 100 Manual Labels:
                   precision    recall  f1-score   support

  qualité produit       0.36      0.20      0.25        41
service livraison       0.25      0.62      0.36        21
   service client       0.15      0.11      0.13        27

        micro avg       0.26      0.27      0.26        89
        macro avg       0.26      0.31      0.25        89
     weighted avg       0.27      0.27      0.24        89
      samples avg       0.20      0.18      0.18        89



c:\Users\vires\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vires\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vires\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is"

In [18]:
# Matrices de confusion

for i, label in enumerate(labels):
    cm = confusion_matrix(y_avis_true[:, i], y_avis_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"\nMatrice de confusion — {label}")
    display(cm_df)


Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,45,14
Vrai 1,33,8



Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,41,38
Vrai 1,8,13



Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,56,17
Vrai 1,24,3
